# Stochastic Models in Python
## Monte Carlo Simulation of a Hospital Emergency Department

### Scenario
A hospital wants to understand how the number of doctors affects patient waiting times in its emergency department. You will build a discrete-event simulation, run repeated experiments, visualize the results, and make a recommendation supported by evidence.

> Of course, this simplified model is for learning stochastic simulation. It is not suitable for real clinical or staffing decisions.

## Objectives
By completing this lab, you should be able to:
- Model arrivals using a Poisson process.
- Model uncertain service durations using probability distributions.
- Represent queues and limited resources with `simpy`.
- Run Monte Carlo replications and summarize results with `pandas` and `numpy`.
- Visualize results with `matplotlib` and `seaborn`.
- Interpret confidence intervals and discuss model limitations.

## Libraries
- `numpy`: random-number generation and numerical operations
- `simpy`: discrete-event simulation
- `pandas`: data tables and summaries
- `matplotlib` and `seaborn`: visualizations
- `scipy`: statistical confidence intervals

## Model assumptions
- Patients arrive during an 8-hour period.
- Arrivals follow a Poisson process with average rate 10 patients/hour.
- Every patient visits triage first, then a doctor.
- There are 2 triage nurses.
- Triage duration follows a triangular distribution: minimum 2, mode 5, maximum 10 minutes.
- Doctor service duration follows a lognormal distribution with mean approximately 25 minutes and coefficient of variation 0.5.
- Patients arriving before the end of the shift complete their care, even if they leave after the 8-hour arrival period.

**Task 0 before coding:** Think about what makes this a stochastic model.

## Set up the environment

We will need these libraries: <br>
simpy numpy pandas matplotlib seaborn scipy

### Import libraries

Fill in the missing library names.

In [ ]:
import ______ as np
import ______ as pd
import matplotlib.pyplot as plt
import seaborn as sns
import simpy
from scipy import stats

sns.set_theme(style="whitegrid")

### Define the model parameters

Complete the missing values based on the assumptions above.

In [ ]:
ARRIVAL_RATE_PER_HOUR = ____
ARRIVAL_PERIOD_MIN = ____ * ____   # 8 hours, expressed in minutes
TRIAGE_NURSES = ____
TRIAGE_TIME_MIN = (____, ____, ____)  # minimum, mode, maximum
DOCTOR_MEAN_SERVICE_MIN = ____
WAITING_THRESHOLD_MIN = ____

## Create one simulation

### Write the simulation function

Complete the missing pieces in the code. The function should simulate one shift and return a `pandas` DataFrame with one row per patient.

Hints:
- Use `np.random.default_rng(seed)` for reproducible random numbers.
- Create a `simpy.Environment()`.
- Create two resources: triage nurses and doctors.
- Generate the time between arrivals with an exponential distribution. If the arrival rate is λ patients per hour, the mean inter-arrival time in minutes is \(60/λ\).
- For triage duration, use `rng.triangular(minimum, mode, maximum)`.
- Use a lognormal distribution for doctor service time.
- Record arrival time, triage waiting time, doctor waiting time, departure time, and total waiting time.

In [ ]:
def simulate_ed(n_doctors=3, seed=0):
    rng = np.random.default_rng(____)
    env = simpy.____()
    triage = simpy.Resource(env, capacity=____)
    doctors = simpy.Resource(env, capacity=____)
    records = []

    # Lognormal parameters for arithmetic mean 25 minutes and CV 0.5
    cv = 0.5
    sigma = np.sqrt(np.log(1 + cv**2))
    mu = np.log(DOCTOR_MEAN_SERVICE_MIN) - sigma**2 / 2

    def patient(patient_id, arrival_time):
        record = {"patient_id": patient_id, "arrival_min": arrival_time}

        with triage.request() as request:
            yield request
            triage_start = env.now
            record["triage_wait_min"] = triage_start - arrival_time
            triage_duration = rng.triangular(*TRIAGE_TIME_MIN)
            yield env.timeout(____)
            record["triage_duration_min"] = triage_duration

        with doctors.request() as request:
            yield request
            doctor_start = env.now
            record["doctor_wait_min"] = doctor_start - (triage_start + triage_duration)
            service_duration = rng.lognormal(mean=____, sigma=____)
            yield env.timeout(service_duration)
            record["doctor_service_min"] = service_duration
            record["departure_min"] = env.now
            record["total_time_min"] = env.now - arrival_time
            record["total_wait_min"] = (
                record["triage_wait_min"] + record["doctor_wait_min"]
            )
            record["wait_over_threshold"] = (
                record["total_wait_min"] > WAITING_THRESHOLD_MIN
            )
            records.append(record)

    def arrivals():
        patient_id = 0
        while True:
            interarrival = rng.exponential(60 / ARRIVAL_RATE_PER_HOUR)
            yield env.timeout(____)
            if env.now > ARRIVAL_PERIOD_MIN:
                break
            patient_id += 1
            env.process(patient(patient_id, env.now))

    env.process(____)
    env.run()
    return pd.DataFrame(records).sort_values("arrival_min").reset_index(drop=True)

# Test your function:
patients = simulate_ed(n_doctors=____, seed=____)
patients.head()

### Inspect the output

1. How many patients were simulated in this shift? ____________________
2. What does one row of the DataFrame represent? ____________________
3. Why is the random seed useful? _____________________________________

## Calculate performance measures

### Complete the summary function

Calculate:
- number of patients,
- mean and median total waiting time,
- 90th percentile of total waiting time,
- proportion waiting more than the threshold,
- mean total time in the department.

In [ ]:
def summarize_shift(patients):
    return pd.Series({
        "patients_served": len(patients),
        "mean_total_wait_min": patients["total_wait_min"].____(),
        "median_total_wait_min": patients["total_wait_min"].____(),
        "p90_total_wait_min": patients["total_wait_min"].____(____),
        "share_wait_over_threshold": patients["wait_over_threshold"].____(),
        "mean_total_time_min": patients["total_time_min"].____(),
    })

summarize_shift(patients).round(2)

**Question 2:** Why might the 90th percentile be more informative than the mean when describing patient experience?

__________________________________________________________________  
__________________________________________________________________

## Monte Carlo experiment

A single shift is only one random outcome. Repeat the simulation for several independent seeds and compare 2, 3, and 4 doctors.

### Run repeated simulations

Complete the missing values and loop. Use at least 30 replications per staffing scenario.

In [ ]:
N_REPLICATIONS = ____
DOCTOR_SCENARIOS = [____, ____, ____]

rows = []
for n_doctors in DOCTOR_SCENARIOS:
    for rep in range(N_REPLICATIONS):
        patients = simulate_ed(
            n_doctors=n_doctors,
            seed=10_000 * n_doctors + rep
        )
        metrics = summarize_shift(patients).to_dict()
        rows.append({
            "doctors": n_doctors,
            "replication": rep,
            **metrics
        })

results = pd.DataFrame(rows)
results.head()

### Aggregate the results

Group the results by number of doctors and calculate the mean of each performance measure.

In [ ]:
summary = (
    results.groupby(____)[[
        "mean_total_wait_min",
        "median_total_wait_min",
        "p90_total_wait_min",
        "share_wait_over_threshold",
        "patients_served"
    ]]
    .____()
    .reset_index()
)

summary

**Question 3:** As the number of doctors increases, what happens to mean waiting time? Is the change proportional to the number of doctors?

Use your table as evidence.

__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________

## Confidence intervals

### Complete the confidence interval function

Use a Student-t confidence interval for the mean across simulation replications:
\[
\bar{x} \pm t_{0.975,n-1}\frac{s}{\sqrt{n}}
\]
where \(n\) is the number of replications and \(s\) is their sample standard deviation.

In [ ]:
def mean_ci(series, confidence=0.95):
    values = np.asarray(series, dtype=float)
    n = len(values)
    mean = values.____()
    sem = stats.____(values)
    margin = stats.t.ppf((1 + confidence) / 2, df=____) * ____
    return pd.Series({
        "mean": mean,
        "ci_low": mean - margin,
        "ci_high": mean + margin
    })

mean_ci(results[results["doctors"] == 3]["mean_total_wait_min"])

### Compare confidence intervals

Calculate the 95% confidence interval for mean waiting time for each staffing scenario and place the results in a table.

**Question 4:** What does the confidence interval describe? What does it *not* tell you about the realism of the model?

__________________________________________________________________  
__________________________________________________________________

## Visualize the results

### Box plot

Complete the code to compare the distribution of mean waiting times across replications.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=results, x=____, y=____)
plt.xlabel("Number of doctors")
plt.ylabel("Mean waiting time per shift (minutes)")
plt.title("Effect of doctor staffing on waiting time")
plt.tight_layout()
plt.show()

### Long-wait risk

Create a bar plot showing the share of patients waiting more than the threshold for each staffing scenario. Label both axes and add a title.

**Your code:**

In [ ]:
# TODO: Create a plot of share_wait_over_threshold by doctors.
# Hint: sns.barplot(...)

### Confidence interval plot

Plot mean waiting time by number of doctors with 95% confidence interval error bars.

**Your code:**

In [ ]:
# TODO: For each staffing scenario, calculate mean_ci(...)
# Then use plt.errorbar(...) to display means and confidence intervals.

## Model critique and extensions

Answer each question in complete sentences.

1. Why are exponential inter-arrival times appropriate for a homogeneous Poisson process?

   __________________________________________________________________

2. The model assumes a constant arrival rate. How could you model a busy afternoon and a quiet night?

   __________________________________________________________________

3. Could triage become a bottleneck? Describe a test you could run.

   __________________________________________________________________

4. Name two assumptions that limit the model's realism.

   __________________________________________________________________

5. Propose one extension using `simpy.PriorityResource`, different patient acuity levels, or staffing costs.

   __________________________________________________________________


**Conclusion:**

__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________  
__________________________________________________________________